In [2]:
pip install torch

In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

Baixa o dadaset e separa os dados para treinamento e teste:
- Treinamento: usado pela Rede Neural pra encontrar padrões e reajustar os pesos
- Teste: usado no fim do treinamento para medir a taxa de erro e realizar o backpropagation

In [5]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 205kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.82MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 22.8MB/s]


Cria esses data loaders, dividindo os conjuntos de dados em mini-lotes pra acelerar o treinamento, realizando o processamento desses lotes de forma paralela

In [6]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


Criando a Rede Neural, especificando acelerador de computação e descrevendo o processo de propagação direta (cálculo dos pesos)

In [7]:
# accelerator permite rodar o modelo em GPU caso tenha
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

# forward propagation
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Definindo o treinamento do modelo:
- Seleciona a Loss Function (Função de Perda pra cálculo do gradiente e realizar o backpropagation)
- Seleciona o Optimizer (Algoritmo de otimização que pega o erro calculado pela Loss Function e calcula o gradiente pra realizar o Backpropagation)


In [12]:
# Loss Function = Cross Entropy Loss
loss_fn = nn.CrossEntropyLoss()
# Optimizer = Stochastic Gradient Descent
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

# Função de treinamento
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


# Função de teste
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

Loop de treinamento (cada iteração = epoch/época):
- Treina em batches(forward propagation, calcula erro e backpropagation)
- Testa no final do treinamento: acurácia (predição correta/tamanho do dataset) e média de erro (taxa de erro/nº de batches)

In [13]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.288951  [   64/60000]
loss: 2.290347  [ 6464/60000]
loss: 2.263209  [12864/60000]
loss: 2.265996  [19264/60000]
loss: 2.258267  [25664/60000]
loss: 2.208829  [32064/60000]
loss: 2.235441  [38464/60000]
loss: 2.188279  [44864/60000]
loss: 2.181279  [51264/60000]
loss: 2.158382  [57664/60000]
Test Error: 
 Accuracy: 43.9%, Avg loss: 2.154722 

Epoch 2
-------------------------------
loss: 2.155869  [   64/60000]
loss: 2.154186  [ 6464/60000]
loss: 2.090290  [12864/60000]
loss: 2.112935  [19264/60000]
loss: 2.063887  [25664/60000]
loss: 1.992943  [32064/60000]
loss: 2.030553  [38464/60000]
loss: 1.946822  [44864/60000]
loss: 1.944517  [51264/60000]
loss: 1.873411  [57664/60000]
Test Error: 
 Accuracy: 57.5%, Avg loss: 1.880671 

Epoch 3
-------------------------------
loss: 1.902057  [   64/60000]
loss: 1.878146  [ 6464/60000]
loss: 1.757781  [12864/60000]
loss: 1.806535  [19264/60000]
loss: 1.690601  [25664/60000]
loss: 1.638668  [32064/600

Salva o modelo: serializa como dicionário, salvando o estado do modelo e de seus parâmetros

In [14]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


Carrega o modelo com seus parâmetros/pesos já treinados

In [15]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

Exemplo de uso do modelo pra classificação multi-classe

In [16]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
